In [32]:
import os
import ee
import json
import time
import geemap
import configparser
import geopandas as gpd

In [16]:
BASE_DIR = os.getcwd()
CONFIG = configparser.ConfigParser()
CONFIG.read(os.path.join(BASE_DIR, 'script_config.ini'))

BASE_PATH = os.path.abspath(os.path.join(os.getcwd(), '..', 'data'))

DATA_RAW = os.path.join(BASE_PATH, 'raw')
DATA_RESULTS = os.path.join(BASE_PATH, '..', 'results')

First we initialize Google Earth Engine

In [17]:
ee.Authenticate()
ee.Initialize()

And then define our area of interest.

In [30]:
shapefile_path = os.path.join(DATA_RAW, 'Tororo_boundary.shp')
gdf = gpd.read_file(shapefile_path)
geojson_file = json.loads(gdf.to_json())
shapefile = ee.FeatureCollection(geojson_file)
roi = shapefile.geometry()

We now download Landsat 8 images from Google Earth Engine from 2015 to 2025. We removes scenes with more than 20% cloud cover before any processing.

In [19]:
collection = (
    ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
    .filterBounds(roi)
    .filterDate("2015-01-01", "2025-12-31")
    .filter(ee.Filter.lt("CLOUD_COVER",20))
)

We also remove cloudy pixels

In [20]:
def mask_landsat(img):
    
    qa = img.select('QA_PIXEL')
    mask = qa.bitwiseAnd(1 << 3).eq(0)
    return img.updateMask(mask)

Next, we create a function for computing indices that we will use to predict mosquito habitat suitability, seasonal outbreak risk and irrigation-driven vector expansion. The indices are as follows;

    1. NDVI (Normalized Difference Vegetation Index) - Mosquito hideouts and habitat
    2. NDWI (Normalized Difference Water Index) - Mosquito breeding
    3. NDBI (Normalized Difference Built-up Index ) - 
    4. LST - Favorable ecological condition for mosquito breeding

In [21]:
def add_indices(img):

    ndvi = img.normalizedDifference(['SR_B5','SR_B4']).rename('NDVI')

    ndwi = img.normalizedDifference(['SR_B3','SR_B5']).rename('NDWI')

    ndbi = img.normalizedDifference(['SR_B6','SR_B5']).rename('NDBI')

    lst = img.select('ST_B10').multiply(0.00341802).add(149.0).rename('LST')

    return img.addBands([ndvi, ndwi, ndbi, lst])

We then apply the processes to our imagery.

In [25]:
collection = (collection.map(mask_landsat).map(add_indices))

Our deep learning foundation models works better with regular time intervals such as months. Therefore, we convert Landsat measurements to monthly averages.

In [26]:
def monthly_composite(year, month):

    start = ee.Date.fromYMD(year, month, 1)
    end = start.advance(1, 'month')

    image = (
        collection
        .filterDate(start, end)
        .mean()
        .select(['NDVI','NDWI','NDBI','LST'])
    )

    return image.set('system:time_start', start.millis())

And then generate time series for 2015-01 to 2025-12 translating to 132 monthly images.

In [27]:
years = list(range(2015, 2026))
months = list(range(1, 13))

images = []

for y in years:
    for m in months:
        images.append(monthly_composite(y, m))

monthly = ee.ImageCollection(images)

Instead of exporting each image separately, we stack them into one multi-band cube consisting of 132 months × 4 predictors = 528 bands.

In [28]:
cube = monthly.toBands()

We then process our data and export the single band image cube to the Drive.

In [ ]:
task = ee.batch.Export.image.toDrive(
    image = cube,
    description = 'andsat_predictor_cube_2015_2025',
    folder = 'Landsat Cube',
    region = roi,
    scale = 30,
    maxPixels = 1e13)

task.start()

print('Export task started...')
while task.active():
    
    status = task.status()
    print(f"Task state: {status['state']} | Time: {time.strftime('%H:%M:%S')}")
    time.sleep(30) 

print('Task finished.')
print('Final status:', task.status())

Export task started...
Task state: READY | Time: 14:52:46
Task state: READY | Time: 14:53:17
Task state: READY | Time: 14:53:47
Task state: READY | Time: 14:54:17
Task state: READY | Time: 14:54:48
Task state: READY | Time: 14:55:18
Task state: READY | Time: 14:55:49
Task state: READY | Time: 14:56:19
Task state: READY | Time: 14:56:50
Task state: READY | Time: 14:57:20
Task state: READY | Time: 14:57:50
Task state: READY | Time: 14:58:21
Task state: READY | Time: 14:58:52
Task state: READY | Time: 14:59:22
Task state: READY | Time: 15:00:56
Task state: READY | Time: 15:01:27
Task state: READY | Time: 15:02:00
Task state: READY | Time: 15:02:30
Task state: READY | Time: 15:03:01
Task state: READY | Time: 15:03:31
Task state: READY | Time: 15:04:02
Task state: READY | Time: 15:04:32
Task state: READY | Time: 15:05:02
Task state: READY | Time: 15:05:32
Task state: READY | Time: 15:06:03
Task state: READY | Time: 15:06:34
Task state: READY | Time: 15:07:04
Task state: READY | Time: 15:07: